# Batch run 
Uruchamia wszystkie 24 configi: 4 modele x 2 warianty x 3 seedy
**Przed startem:** Runtime -> Change runtime type -> **T4 GPU**.
**Po rozlaczeniu Colab:** odpal notebook ponownie z `--skip-existing` (jest w opcjach) - wznawia od pierwszego niewykonanego runa.

## 1. Setup (clone + install)

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
!pip install -q timm==1.0.11 PyYAML==6.0.2

## 2. Dane Pets

In [ ]:
from pathlib import Path
if not Path('data/raw/oxford-iiit-pet/images').exists():
    !python scripts/download_pets.py
else:
    print('Pets juz pobrane.')

## 3. Regeneracja configow

In [ ]:
!python scripts/generate_configs.py

## 4. GPU sanity check

In [ ]:
from src.utils.device import get_device, device_info
d = get_device()
print('Device:', device_info(d))
assert d.type == 'cuda', 'GPU nie aktywny! Runtime -> Change runtime type -> T4 GPU.'

## 5. Dry-run

In [ ]:
!python scripts/run_all.py --overrides configs/colab.yaml --skip-existing --dry-run

## 6. **WLASCIWY BATCH RUN**

`--skip-existing` pomija runy ktore juz maja `final_results.json` - bezpiecznie wznawia po rozlaczeniu Colaba

Mozna filtrowac:
- `--pattern 'A_*.yaml'` - tylko wariant A (12 runow)
- `--pattern 'B_*.yaml'` - tylko wariant B (12 runow)
- `--pattern '*_resnet18_*.yaml'` - tylko ResNet-18 (6 runow)
- `--pattern '*_seed0.yaml'` - tylko seed 0 (8 runow, dobry quick sanity)

Domyslnie: wszystkie 24

In [ ]:
!python scripts/run_all.py --overrides configs/colab.yaml --skip-existing

## 7. Tabela zbiorcza wynikow

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for fr in sorted(Path('outputs').glob('*/final_results.json')):
    d = json.loads(fr.read_text())
    rows.append({
        'run_name': d['run_name'],
        'variant': d['variant'],
        'model': d['model'],
        'seed': d['seed'],
        'best_epoch': d['best_epoch'],
        'best_val_acc': round(d['best_val_acc'], 4),
        'test_acc': round(d['test']['accuracy'], 4),
        'test_bal_acc': round(d['test']['balanced_accuracy'], 4),
        'test_macro_f1': round(d['test']['macro_f1'], 4),
    })
df = pd.DataFrame(rows)
print(f'Runy zakonczone: {len(df)}/24')
df

## 8. Srednia per (wariant, model) ze slupkiem bledu po seedach

In [ ]:
if len(df) > 0:
    agg = df.groupby(['variant', 'model']).agg(
        test_acc_mean=('test_acc', 'mean'),
        test_acc_std=('test_acc', 'std'),
        n_seeds=('seed', 'count'),
    ).round(4)
    print(agg)